# Plate Raster + Synchrony Viewer

Visualizes burst detection results as a 4×6 plate grid showing raster plots overlaid with synchrony signals (sharp + smooth) for each well.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from pipeline_manager import PipelineManager
from config_manager import ConfigManager
from pipeline_tasks import PlateViewerTask

In [ ]:
# Config setup — generates template if it doesn't exist
CONFIG_FILE = Path("../config/default_tasks_params.json")

cm = ConfigManager()
cm.register_task(PlateViewerTask)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Config template written to {CONFIG_FILE}.\n"
        "Edit it to set correct paths for burst_detection_root, curation_output_root, figures_root, etc.\n"
        "Then re-run this cell."
    )

cm.load(CONFIG_FILE)
print(f"Config loaded from {CONFIG_FILE}")

In [ ]:
# Select recording to visualize
RECORDING_KEY = "CX138/260329/T003346/Network/000029"
print(f"Recording: {RECORDING_KEY}")

In [ ]:
# Register and run plate viewer task
pipeline_mgr = PipelineManager()
pipeline_mgr.register_task(PlateViewerTask)

# Plate-level task: registered once with sentinel well_id
pipeline_mgr.add_well(RECORDING_KEY, "__plate__")

# Get and run pending task
pending = list(pipeline_mgr.get_pending_tasks())
if not pending:
    print("No pending tasks.")
else:
    for task_cls, rec_key, well_id in pending:
        print(f"Running {task_cls.task_name} for {rec_key} ...")
        params = cm.get_config(task_cls.task_name, rec_key, well_id)
        task = task_cls()
        output = task.run(rec_key, well_id, data_path=None, params=params)
        pipeline_mgr.update_status(task_cls.task_name, rec_key, well_id, "complete")
        print(f"✓ Saved: {output}")

In [ ]:
# Display the generated HTML
from IPython.display import IFrame

figures_root = Path(cm.get_task_params("plate_viewer").get("figures_root", "./figures"))
html_path = figures_root / RECORDING_KEY / "plate_viewer.html"

if html_path.exists():
    print(f"Displaying: {html_path}")
    IFrame(src=str(html_path), width="100%", height=800)
else:
    print(f"HTML not found: {html_path}")